# Notebook 1 – Data Exploration

This notebook gives a first look at the toy store e-commerce dataset:
- Dataset shape and dtypes
- Null / duplicate checks
- Descriptive statistics
- Key business metrics (total revenue, orders, customers)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data_loader import load_or_generate, build_master

# Load (or auto-generate) the dataset
tables = load_or_generate('../data', seed=42)
master = build_master(tables)

customers   = tables['customers']
products    = tables['products']
orders      = tables['orders']
order_items = tables['order_items']
print('Dataset loaded.')

## 1. Table Shapes

In [ ]:
for name, df in tables.items():
    print(f'{name:15s}: {df.shape[0]:>7,} rows × {df.shape[1]} cols')

## 2. Schema Overview

In [ ]:
for name, df in tables.items():
    print(f'\n── {name} ──')
    display(df.dtypes.to_frame('dtype'))

## 3. Missing Values & Duplicates

In [ ]:
print('Null counts in master table:')
display(master.isnull().sum().to_frame('nulls'))

print('\nDuplicate rows in master:', master.duplicated().sum())

## 4. Descriptive Statistics

In [ ]:
display(master[['unit_price','quantity','revenue','age']].describe().round(2))

## 5. Key Business KPIs

In [ ]:
delivered = master[master['is_cancelled'] == False]

kpis = {
    'Total Revenue (USD)':    f"${delivered['revenue'].sum():,.2f}",
    'Total Orders':           f"{delivered['order_id'].nunique():,}",
    'Total Customers':        f"{delivered['customer_id'].nunique():,}",
    'Total Products':         f"{delivered['product_id'].nunique():,}",
    'Avg. Order Value (USD)': f"${delivered.groupby('order_id')['revenue'].sum().mean():,.2f}",
    'Cancellation Rate':      f"{master['is_cancelled'].mean()*100:.1f}%",
}

for k, v in kpis.items():
    print(f'{k:<30} {v}')

## 6. Category & Product Counts

In [ ]:
print('Products per category:')
display(products['category'].value_counts())

print('\nOrder status breakdown:')
display(orders['status'].value_counts(normalize=True).mul(100).round(1).to_frame('%'))